# Date and Time Functions

## Overview
Dates and times are among the most error-prone data types in SQL — different databases handle them differently, timezone handling is subtle, and arithmetic has many edge cases.

**SQLite date/time storage:**
SQLite has no native DATE or TIMESTAMP type. Dates are stored as TEXT (ISO 8601), INTEGER (Unix epoch seconds), or REAL (Julian day number). The date functions accept all three formats.

| SQLite Function | Returns | Notes |
|---|---|---|
| `DATE(s)` | `YYYY-MM-DD` | Parses ISO text, Julian, or epoch |
| `TIME(s)` | `HH:MM:SS` | Time component |
| `DATETIME(s)` | `YYYY-MM-DD HH:MM:SS` | Full datetime |
| `STRFTIME(fmt, s)` | Formatted string | Most flexible |
| `JULIANDAY(s)` | Real number | Days since Nov 24, 4714 BC; useful for differences |
| `'now'` | Current UTC datetime | Use `'localtime'` modifier for local time |

**Common STRFTIME format codes:** `%Y` year, `%m` month, `%d` day, `%H` hour, `%M` minute, `%W` week-of-year, `%j` day-of-year

**PostgreSQL native date/time types:**
`DATE`, `TIME`, `TIMESTAMP`, `TIMESTAMPTZ` (with timezone — recommended for events), `INTERVAL`. Arithmetic uses operators: `date + INTERVAL '3 days'`, `date1 - date2`. Functions: `EXTRACT()`, `DATE_TRUNC()`, `AGE()`, `NOW()`, `TO_DATE()`, `TO_TIMESTAMP()`.

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cur  = conn.cursor()

cur.executescript("""
CREATE TABLE admissions (
    adm_id        INTEGER PRIMARY KEY,
    patient_id    INTEGER,
    dept          TEXT,
    admitted_dt   TEXT,        -- YYYY-MM-DD HH:MM
    discharged_dt TEXT,        -- NULL if still admitted
    dob           TEXT         -- patient date of birth YYYY-MM-DD
);

INSERT INTO admissions VALUES
  (1,  1,  'Cardiology',   '2023-02-01 09:30', '2023-02-05 14:00', '1985-03-12'),
  (2,  2,  'Emergency',    '2023-03-15 22:10', '2023-03-18 11:30', '1972-11-04'),
  (3,  3,  'ICU',          '2023-04-20 08:00', NULL,               '1990-07-22'),
  (4,  4,  'Orthopaedics', '2023-06-01 14:00', '2023-06-03 10:15', '1955-01-30'),
  (5,  5,  'Emergency',    '2023-07-10 17:45', '2023-07-11 09:00', '2001-09-15'),
  (6,  6,  'Oncology',     '2023-09-05 11:20', '2023-09-12 16:00', '1968-06-08'),
  (7,  1,  'Cardiology',   '2023-11-22 08:00', '2023-11-25 12:30', '1985-03-12'),
  (8,  9,  'Cardiology',   '2024-01-08 10:15', '2024-01-14 09:00', '1977-08-25'),
  (9,  10, 'Emergency',    '2024-02-14 03:30', '2024-02-14 18:00', '1993-05-19');
""")
conn.commit()

print('admissions table ready:')
print(pd.read_sql('SELECT * FROM admissions', conn).to_string(index=False))


---
## Extracting date parts with STRFTIME


In [ ]:
# Extract individual date/time components
print('=== Date part extraction ===')
q = """
SELECT adm_id,
       admitted_dt,
       DATE(admitted_dt)                   AS date_only,
       TIME(admitted_dt)                   AS time_only,
       STRFTIME('%Y',  admitted_dt)        AS year,
       STRFTIME('%m',  admitted_dt)        AS month,
       STRFTIME('%d',  admitted_dt)        AS day,
       STRFTIME('%H',  admitted_dt)        AS hour,
       STRFTIME('%W',  admitted_dt)        AS week_of_year,
       STRFTIME('%Y-%m', admitted_dt)      AS year_month
FROM   admissions
ORDER BY admitted_dt
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print('=== PostgreSQL equivalents ===')
print("""
  EXTRACT(YEAR  FROM admitted_dt)    -- integer: 2023
  EXTRACT(MONTH FROM admitted_dt)    -- integer: 3
  EXTRACT(HOUR  FROM admitted_dt)    -- integer: 22
  DATE_TRUNC('month', admitted_dt)   -- truncate to 2023-03-01 00:00:00
  DATE_TRUNC('year',  admitted_dt)   -- truncate to 2023-01-01 00:00:00
  TO_CHAR(admitted_dt, 'YYYY-MM')    -- formatted string: '2023-03'
""")


---
## Date arithmetic and length of stay


In [ ]:
# Length of stay in days using JULIANDAY
print('=== Length of stay (days) ===')
q = """
SELECT adm_id,
       patient_id,
       dept,
       DATE(admitted_dt)    AS admit_date,
       DATE(discharged_dt)  AS discharge_date,
       CASE
           WHEN discharged_dt IS NULL
               THEN '(still admitted)'
           ELSE
               CAST(
                   CAST(JULIANDAY(discharged_dt) - JULIANDAY(admitted_dt) AS INTEGER)
                   AS TEXT) || ' days'
       END AS los
FROM   admissions
ORDER BY adm_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Hours in ICU / short stay
print()
print('=== Length of stay in hours (for short admissions) ===')
q2 = """
SELECT adm_id, dept,
       ROUND((JULIANDAY(discharged_dt) - JULIANDAY(admitted_dt)) * 24, 1) AS los_hours
FROM   admissions
WHERE  discharged_dt IS NOT NULL
ORDER BY los_hours
"""
print(pd.read_sql(q2, conn).to_string(index=False))

print()
print('=== PostgreSQL date arithmetic ===')
print("""
  discharged_dt::DATE - admitted_dt::DATE              -- integer days
  discharged_dt - admitted_dt                          -- interval '4 days 04:30:00'
  EXTRACT(EPOCH FROM (discharged_dt - admitted_dt))/3600  -- total hours as float
  admitted_dt + INTERVAL '30 days'                     -- add 30 days
""")


---
## Age calculation and date filtering


In [ ]:
# Age at admission (correct method accounting for birthday not yet reached)
print('=== Patient age at admission ===')
q = """
SELECT adm_id,
       patient_id,
       dob,
       DATE(admitted_dt) AS admit_date,
       CAST(
           (STRFTIME('%Y', admitted_dt) - STRFTIME('%Y', dob))
           - CASE
               WHEN STRFTIME('%m-%d', admitted_dt) < STRFTIME('%m-%d', dob)
               THEN 1 ELSE 0
             END
       AS INTEGER) AS age_at_admission
FROM   admissions
ORDER BY age_at_admission
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Date range filtering — correct approach for datetime columns
print()
print('=== Admissions in Q3 2023 (safe range filter) ===')
q2 = """
SELECT adm_id, patient_id, dept,
       DATE(admitted_dt) AS admit_date,
       DATE(discharged_dt) AS discharge_date
FROM   admissions
WHERE  admitted_dt >= '2023-07-01'
  AND  admitted_dt <  '2023-10-01'    -- < start of Q4, not <= end of Q3
ORDER BY admitted_dt
"""
print(pd.read_sql(q2, conn).to_string(index=False))


---
## Period bucketing and time-series aggregation


In [ ]:
# Monthly admission counts — common in healthcare reporting
print('=== Monthly admissions ===')
q = """
SELECT STRFTIME('%Y-%m', admitted_dt)  AS month,
       dept,
       COUNT(*)                         AS admissions,
       SUM(CASE WHEN discharged_dt IS NULL THEN 1 ELSE 0 END) AS still_admitted
FROM   admissions
GROUP BY month, dept
ORDER BY month, dept
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Weekend vs weekday
print()
print('=== Weekend vs weekday admissions ===')
q2 = """
SELECT
    CASE STRFTIME('%w', admitted_dt)  -- 0=Sunday, 6=Saturday
        WHEN '0' THEN 'Weekend'
        WHEN '6' THEN 'Weekend'
        ELSE          'Weekday'
    END AS day_type,
    COUNT(*) AS admissions
FROM   admissions
GROUP BY day_type
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# Days since admission (for current patients)
print()
print('=== Time in hospital for current admissions (as of 2024-03-01) ===')
q3 = """
SELECT adm_id, dept, admitted_dt,
       CAST(JULIANDAY('2024-03-01') - JULIANDAY(admitted_dt) AS INTEGER) AS days_in_hospital
FROM   admissions
WHERE  discharged_dt IS NULL
"""
print(pd.read_sql(q3, conn).to_string(index=False))

conn.close()


---
## Common Pitfalls

**1. Filtering on a datetime column with a date-only string misses part of the day**
`WHERE admitted_dt = '2023-02-01'` will not match `'2023-02-01 09:30'` in SQLite because string equality requires an exact match. Use a range: `WHERE admitted_dt >= '2023-02-01' AND admitted_dt < '2023-02-02'`, or `WHERE DATE(admitted_dt) = '2023-02-01'`.

**2. JULIANDAY differences are fractional, not integer days**
`JULIANDAY(discharged) - JULIANDAY(admitted)` returns a REAL (e.g. 4.1875 days). Casting to INTEGER truncates — decide explicitly whether to truncate (floor) or round based on your definition of length of stay.

**3. Age from year difference alone is wrong before the birthday**
`STRFTIME('%Y','now') - STRFTIME('%Y', dob)` overcounts by 1 for anyone who hasn't reached their birthday yet this year. Always include the month-day comparison shown in this notebook.

**4. `DATE('now')` returns UTC, not local time**
In Canada, `DATE('now')` returns tomorrow's date after approximately 7–8 PM Atlantic time (UTC-4 in summer). Use `DATE('now', 'localtime')` for local dates, or manage timezone conversion in your application layer. In PostgreSQL, use `NOW() AT TIME ZONE 'America/Halifax'` or store all timestamps as `TIMESTAMPTZ`.

**5. Timezone-naive TIMESTAMP in PostgreSQL silently loses timezone information**
`TIMESTAMP` stores no timezone — inserting `2023-03-15 22:10:00 EST` strips the timezone and stores only the time as entered. Use `TIMESTAMPTZ` for any real-world event timestamps to ensure correct UTC storage and timezone-aware retrieval.

**6. BETWEEN on dates is inclusive — `BETWEEN '2023-01-01' AND '2023-03-31'` includes all of March 31**
For datetime columns this means `2023-03-31 23:59:59` is included. Use `>= start AND < next_period_start` for clarity and to correctly handle time components.


---
*sql_methods_library - Samantha McGarrigle*